In [26]:
from huggingface_hub import hf_hub_download
import importlib.util

# Descarga el loader unificado (está en cualquier repo)
loader_path = hf_hub_download("Reynier/dga-cnn", "dga_loader.py")
spec = importlib.util.spec_from_file_location("dga_loader", loader_path)
mod = importlib.util.module_from_spec(spec); spec.loader.exec_module(mod)

name_model="bilstm"
# Carga cualquier modelo
model, m = mod.load_dga_model(name_model)
results = mod.predict_domains(m, model, ["google.com", "xkr3f9mq.ru"])
print(results)

bilstm_best.pth:   0%|          | 0.00/740k [00:00<?, ?B/s]

model.py: 0.00B [00:00, ?B/s]

  bilstm ready.
[{'domain': 'google.com', 'label': 'legit', 'score': 0.0298}, {'domain': 'xkr3f9mq.ru', 'label': 'dga', 'score': 0.9853}]


In [27]:
# Para un solo dominio, pasamos el dominio en una lista
results = mod.predict_domains(m, model, ["google.com"])

# Como devuelve una lista, accedemos al primer elemento [0]
prediccion = results[0]

print(f"Dominio: {prediccion['domain']}")
print(f"Etiqueta: {prediccion['label']}")
print(f"Puntaje: {prediccion['score']}")

Dominio: google.com
Etiqueta: legit
Puntaje: 0.0298


In [30]:
import pandas as pd
import time

families = ['bazarbackdoor.gz',
            'bazarbackdoor_v2.gz',
            'bazarbackdoor_v3.gz',
            'bigviktor.gz',
            'bumblebee.gz',
            'ccleaner.gz',
            'dmsniff.gz',
            'enviserv.gz',
            'fosniw.gz',
            'goz.gz',
            'gozi_rfc4343.gz',
            'infy.gz',
            'monerodownloader.gz',
            'newgoz.gz',
            'ngioweb.gz',
            'pandabanker.gz',
            'pizd.gz',
            'reconyc.gz',
            'sharkbot.gz',
            'szribi.gz',
            'torpig.gz',
            'tufik.gz',
            'verblecon.gz',
            'wd.gz',
            'xshellghost.gz',
           ]

runs = 30
chunk_size = 50
offset_legit = 1500

if model is None:
    print("❌ No se pudo cargar el modelo. Deteniendo evaluación.")
else:
    print("🚀 Iniciando evaluación del modelo...")

    for family in families:
        print(f"📂 Evaluando familia: {family}")

        dga = pd.read_csv(
            f'/content/drive/My Drive/New_Families/{family}',
            chunksize=chunk_size
        )

        legit = pd.read_csv(
            '/content/drive/My Drive/Familias_Test/legit.gz',
            skiprows=range(1, offset_legit + 1),
            chunksize=chunk_size
        )

        for run in range(runs):
            print(f'  🔄 {run+1:02}/{runs}', end='\r')

            try:
                dfw = pd.concat([dga.get_chunk(), legit.get_chunk()], ignore_index=True)
            except StopIteration:
                print(f"\n⚠️ No hay suficientes datos para completar {family}")
                break

            pred = []
            prob = []
            query_time = []

            for domain_to_check in dfw.domain.values:
                st = time.time()

                try:
                    results = mod.predict_domains(m, model, [domain_to_check])
                    result = results[0]

                    label_value = 1 if result['label'] == 'dga' else 0
                    probability = result['score']

                    pred.append(label_value)
                    prob.append(probability)

                except Exception:
                    pred.append(0)
                    prob.append(0.5)

                query_time.append(time.time() - st)

            dfw['pred'] = pred
            dfw['prob'] = prob
            dfw['query_time'] = query_time

            dfw.to_csv(
                f'/content/drive/My Drive/results/results_{name_model}_{family}_{run}.csv.gz',
                index=False
            )

    print("\n✅ Evaluación completada!")

🚀 Iniciando evaluación del modelo...
📂 Evaluando familia: bazarbackdoor.gz
📂 Evaluando familia: bazarbackdoor_v2.gz
📂 Evaluando familia: bazarbackdoor_v3.gz
📂 Evaluando familia: bigviktor.gz
📂 Evaluando familia: bumblebee.gz
📂 Evaluando familia: ccleaner.gz
📂 Evaluando familia: dmsniff.gz
📂 Evaluando familia: enviserv.gz
📂 Evaluando familia: fosniw.gz
📂 Evaluando familia: goz.gz
📂 Evaluando familia: gozi_rfc4343.gz
📂 Evaluando familia: infy.gz
📂 Evaluando familia: monerodownloader.gz
📂 Evaluando familia: newgoz.gz
📂 Evaluando familia: ngioweb.gz
📂 Evaluando familia: pandabanker.gz
📂 Evaluando familia: pizd.gz
📂 Evaluando familia: reconyc.gz
📂 Evaluando familia: sharkbot.gz
📂 Evaluando familia: szribi.gz
📂 Evaluando familia: torpig.gz
📂 Evaluando familia: tufik.gz
📂 Evaluando familia: verblecon.gz
📂 Evaluando familia: wd.gz
📂 Evaluando familia: xshellghost.gz

✅ Evaluación completada!


In [33]:
import os
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix
)
import matplotlib.pyplot as plt
import seaborn as sns

# Montar Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Función para calcular FPR y TPR
def fpr_tpr(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    fpr = fp / (fp + tn + 1e-10)
    tpr = tp / (tp + fn + 1e-10)
    return fpr, tpr

families = ['bazarbackdoor.gz', 'bazarbackdoor_v2.gz', 'bazarbackdoor_v3.gz', 'bigviktor.gz', 'bumblebee.gz', 'ccleaner.gz', 'dmsniff.gz', 'enviserv.gz', 'fosniw.gz', 'goz.gz', 'gozi_rfc4343.gz', 'infy.gz', 'monerodownloader.gz', 'newgoz.gz', 'ngioweb.gz', 'pandabanker.gz', 'pizd.gz', 'reconyc.gz', 'sharkbot.gz', 'szribi.gz', 'torpig.gz', 'tufik.gz', 'verblecon.gz', 'wd.gz', 'xshellghost.gz']
runs = 30

# Listas para métricas globales
all_acc, all_acc_std = [], []
all_pre, all_pre_std = [], []
all_rec, all_rec_std = [], []
all_f1, all_f1_std = [], []
all_fpr, all_fpr_std = [], []
all_tpr, all_tpr_std = [], []
all_qt, all_qts = [], []
total_unknowns_global = 0

for family in families:
    acc, pre, rec, f1, fpr, tpr, qt = [], [], [], [], [], [], []
    total_unknowns = 0

    for run in range(runs):
        path = f'/content/drive/My Drive/results/results_{name_model}_{family}_{run}.csv.gz'
        if not os.path.exists(path): continue

        df = pd.read_csv(path)
        y_true = (df['label'] == 'dga').astype(int)
        y_pred = df['pred']

        acc.append(accuracy_score(y_true, y_pred))
        pre.append(precision_score(y_true, y_pred, zero_division=0))
        rec.append(recall_score(y_true, y_pred, zero_division=0))
        f1.append(f1_score(y_true, y_pred, zero_division=0))

        fpr_val, tpr_val = fpr_tpr(y_true, y_pred)
        fpr.append(fpr_val)
        tpr.append(tpr_val)
        if 'query_time' in df.columns: qt.append(df['query_time'].mean())

    if acc:
        print(f'{family.split(".")[0]:15}: '
              f'acc:{np.mean(acc):.2f}±{np.std(acc):.3f} '
              f'f1:{np.mean(f1):.2f}±{np.std(f1):.3f} '
              f'pre:{np.mean(pre):.2f}±{np.std(pre):.3f} '
              f'rec:{np.mean(rec):.2f}±{np.std(rec):.3f} '
              f'FPR:{np.mean(fpr):.2f}±{np.std(fpr):.3f} '
              f'TPR:{np.mean(tpr):.2f}±{np.std(tpr):.3f} '
              f'QT:{np.mean(qt):.5f}±{np.std(qt):.5f}')

        all_acc.append(np.mean(acc))
        all_acc_std.append(np.std(acc))
        all_pre.append(np.mean(pre))
        all_pre_std.append(np.std(pre))
        all_rec.append(np.mean(rec))
        all_rec_std.append(np.std(rec))
        all_f1.append(np.mean(f1))
        all_f1_std.append(np.std(f1))
        all_fpr.append(np.mean(fpr))
        all_fpr_std.append(np.std(fpr))
        all_tpr.append(np.mean(tpr))
        all_tpr_std.append(np.std(tpr))
        all_qt.append(np.mean(qt))
        all_qts.append(np.std(qt))
        total_unknowns_global += total_unknowns

print("\n### ✅ Métricas recolectadas correctamente ###")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
bazarbackdoor  : acc:0.94±0.027 f1:0.94±0.027 pre:0.93±0.031 rec:0.95±0.033 FPR:0.07±0.032 TPR:0.95±0.033 QT:0.00363±0.00066
bazarbackdoor_v2: acc:0.95±0.020 f1:0.95±0.019 pre:0.93±0.029 rec:0.97±0.024 FPR:0.07±0.032 TPR:0.97±0.024 QT:0.00363±0.00055
bazarbackdoor_v3: acc:0.72±0.041 f1:0.64±0.073 pre:0.88±0.049 rec:0.51±0.087 FPR:0.07±0.032 TPR:0.51±0.087 QT:0.00358±0.00077
bigviktor      : acc:0.52±0.028 f1:0.17±0.074 pre:0.59±0.185 rec:0.10±0.048 FPR:0.07±0.032 TPR:0.10±0.048 QT:0.00326±0.00034
bumblebee      : acc:0.95±0.017 f1:0.95±0.017 pre:0.93±0.029 rec:0.97±0.022 FPR:0.07±0.032 TPR:0.97±0.022 QT:0.00369±0.00049
ccleaner       : acc:0.78±0.038 f1:0.73±0.057 pre:0.90±0.043 rec:0.62±0.074 FPR:0.07±0.032 TPR:0.62±0.074 QT:0.00376±0.00093
dmsniff        : acc:0.91±0.095 f1:0.89±0.137 pre:0.92±0.054 rec:0.89±0.185 FPR:0.07±0.032 TPR:0.89±0.185 QT:0.00342±0.

In [34]:
import pandas as pd
import numpy as np

# Crear el DataFrame usando todas las listas de medias y desviaciones
metrics_dict = {
    'Family': [f.split('.')[0] for f in families],
    'Accuracy': all_acc,
    'Acc_std': all_acc_std,
    'Precision': all_pre,
    'Pre_std': all_pre_std,
    'Recall': all_rec,
    'Rec_std': all_rec_std,
    'F1-Score': all_f1,
    'F1_std': all_f1_std,
    'FPR': all_fpr,
    'FPR_std': all_fpr_std,
    'TPR': all_tpr,
    'TPR_std': all_tpr_std,
    'Query_Time_Mean': all_qt,
    'Query_Time_Std': all_qts
}

df_metrics = pd.DataFrame(metrics_dict)

# Calcular fila de promedios globales
global_means = df_metrics.mean(numeric_only=True).to_dict()
global_means['Family'] = 'GLOBAL_MEAN'

df_metrics = pd.concat([df_metrics, pd.DataFrame([global_means])], ignore_index=True)

output_path = f'/content/drive/My Drive/results/metricas_globales_final_{name_model}.csv'
df_metrics.to_csv(output_path, index=False)

print(f"✅ CSV final con todas las desviaciones guardado en: {output_path}")
display(df_metrics)

✅ CSV final con todas las desviaciones guardado en: /content/drive/My Drive/results/metricas_globales_final_bilstm.csv


,Family,Accuracy,Acc_std,Precision,Pre_std,Recall,Rec_std,F1-Score,F1_std,FPR,FPR_std,TPR,TPR_std,Query_Time_Mean,Query_Time_Std
0,bazarbackdoor,0.939333,0.027439,0.931786,0.030830,0.948667,0.033340,0.939858,0.027362,0.07,0.032146,0.948667,0.033340,0.003634,0.000662
1,bazarbackdoor_v2,0.951333,0.019788,0.933725,0.028782,0.972667,0.023935,0.952439,0.019296,0.07,0.032146,0.972667,0.023935,0.003626,0.000549
2,bazarbackdoor_v3,0.717667,0.041286,0.880333,0.048683,0.505333,0.087168,0.637402,0.072717,0.07,0.032146,0.505333,0.087168,0.003578,0.000770
3,bigviktor,0.516333,0.028223,0.585156,0.184873,0.102667,0.047814,0.172146,0.074082,0.07,0.032146,0.102667,0.047814,0.003258,0.000343
4,bumblebee,0.949667,0.017221,0.933647,0.028745,0.969333,0.021746,0.950750,0.016560,0.07,0.032146,0.969333,0.021746,0.003687,0.000487
5,ccleaner,0.775667,0.038182,0.900278,0.042779,0.621333,0.073563,0.732447,0.056511,0.07,0.032146,0.621333,0.073563,0.003757,0.000934
6,dmsniff,0.908000,0.094918,0.920701,0.053927,0.886000,0.184691,0.893640,0.137101,0.07,0.032146,0.886000,0.184691,0.003421,0.000452
7,enviserv,0.822333,0.032216,0.911984,0.038987,0.714667,0.058180,0.799868,0.041188,0.07,0.032146,0.714667,0.058180,0.003340,0.000422
8,fosniw,0.465000,0.016073,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.07,0.032146,0.000000,0.000000,0.003658,0.000655
9,goz,0.965000,0.016073,0.935427,0.028238,1.000000,0.000000,0.966417,0.015035,0.07,0.032146,1.000000,0.000000,0.003681,0.000477
